<a href="https://colab.research.google.com/github/walruto/FDM-Research/blob/main/FBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/FDM PROJECT/malimg_dataset.zip" -d /content/malimg

replace /content/malimg/malimg_paper_dataset_imgs/Adialer.C/000bde2e9a94ba41c0c111ffd80647c2.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls /content/malimg

malimg_paper_dataset_imgs


In [ ]:
!find /content/malimg -type f | sed 's/.*\.//' | sort | uniq -c

   9339 png
      1 txt


In [ ]:
!cat /content/malimg/malimg_paper_dataset_imgs/malimg_dataset_readme.txt

The dataset contains the images of malware family names as mentioned in Tab.3 of the paper "Malware Images: Visualization and Automatic Classification" (http://dl.acm.org/citation.cfm?id=2016908)
If this dataset is used, the above paper must be cited.
This dataset is not be distributed.
Any questions, contact lakshmanan_nataraj@umail.ucsb.edu
Oct. 2013

In [ ]:
from pathlib import Path

root = Path("/content/malimg/malimg_paper_dataset_imgs")
folders = [p for p in root.iterdir() if p.is_dir()]
print(len(folders))

25


In [ ]:
counts = {}
for f in folders:
    counts[f.name] = len(list(f.glob("*.png")))

sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
for name, n in sorted_counts:
    print(name, n)
print(sum(counts.values()))

Allaple.A 2949
Allaple.L 1591
Yuner.A 800
Instantaccess 431
VB.AT 408
Fakerean 381
Lolyda.AA1 213
C2LOP.gen!g 200
Alueron.gen!J 198
Lolyda.AA2 184
Dialplatform.B 177
Dontovo.A 162
Lolyda.AT 159
Rbot!gen 158
C2LOP.P 146
Obfuscator.AD 142
Malex.gen!J 136
Swizzor.gen!I 132
Swizzor.gen!E 128
Lolyda.AA3 123
Adialer.C 122
Agent.FYI 116
Autorun.K 106
Wintrim.BX 97
Skintrim.N 80
9339


In [ ]:
from PIL import Image

sizes = set()
for f in folders:
    for p in f.glob("*.png"):
        with Image.open(p) as im:
            sizes.add(im.size)

print(len(sizes))
widths  = [w for w, h in sizes]
heights = [h for w, h in sizes]
print(min(widths), max(widths), min(heights), max(heights))

835
64 1024 208 5334


In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image

root = Path("/content/malimg/malimg_paper_dataset_imgs")
folders = sorted(p for p in root.iterdir() if p.is_dir())

class_names = [f.name for f in folders]
label_of = {name: i for i, name in enumerate(class_names)}

X, y = [], []
for f in folders:
    for p in sorted(f.glob("*.png")):
        with Image.open(p) as im:
            im = im.convert("L")
            im = im.resize((64, 64), resample=Image.Resampling.BILINEAR)
            X.append(np.asarray(im, dtype=np.uint8))
        y.append(label_of[f.name])

X = np.stack(X)
y = np.array(y)
print(X.shape, y.shape, X.dtype)

np.savez_compressed(
    "/content/drive/MyDrive/FDM PROJECT/malimg_64.npz",
    X=X, y=y, class_names=np.array(class_names),
)
print("saved")

(9339, 64, 64) (9339,) uint8
saved


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
idx = np.arange(len(y))

train_idx, temp_idx = train_test_split(
    idx, test_size=0.40, stratify=y, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=SEED
)

print(len(train_idx), len(val_idx), len(test_idx))
assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)

print(f"{'family':16s} train  val  test")
for i, name in enumerate(class_names):
    print(f"{name:16s} {(y[train_idx]==i).sum():5d} {(y[val_idx]==i).sum():4d} {(y[test_idx]==i).sum():5d}")

np.savez("/content/drive/MyDrive/FDM PROJECT/split_seed42.npz",
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

5603 1868 1868
family           train  val  test
Adialer.C           73   25    24
Agent.FYI           70   23    23
Allaple.A         1769  590   590
Allaple.L          954  319   318
Alueron.gen!J      119   40    39
Autorun.K           64   21    21
C2LOP.P             88   29    29
C2LOP.gen!g        120   40    40
Dialplatform.B     106   35    36
Dontovo.A           97   32    33
Fakerean           229   76    76
Instantaccess      258   87    86
Lolyda.AA1         128   42    43
Lolyda.AA2         110   37    37
Lolyda.AA3          74   24    25
Lolyda.AT           95   32    32
Malex.gen!J         82   27    27
Obfuscator.AD       85   29    28
Rbot!gen            95   31    32
Skintrim.N          48   16    16
Swizzor.gen!E       77   26    25
Swizzor.gen!I       79   26    27
VB.AT              245   82    81
Wintrim.BX          58   19    20
Yuner.A            480  160   160


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.config.list_physical_devices("GPU"))
keras.utils.set_random_seed(42)

def prep(ix):
    return (X[ix].astype("float32") / 255.0)[..., None]   # scale to 0-1, add channel axis

Xtr, Xva = prep(train_idx), prep(val_idx)
ytr, yva = y[train_idx], y[val_idx]

model = keras.Sequential([
    layers.Input((64, 64, 1)),
    layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(25, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

history = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                    epochs=3, batch_size=64)

[]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       401,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         1,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 407,897 (1.56 MB)

 Trainable params: 407,897 (1.56 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
88/88 ━━━━━━━━━━━━━━━━━━━━ 24s 246ms/step - accuracy: 0.5278 - loss: 1.6888 - val_accuracy: 0.8656 - val_loss: 0.6711
Epoch 2/3
88/88 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.9029 - loss: 0.3803 - val_accuracy: 0.9325 - val_loss: 0.2698
Epoch 3/3
88/88 ━━━━━━━━━━━━━━━━━━━━ 21s 194ms/step - accuracy: 0.9465 - loss: 0.1996 - val_accuracy: 0.9481 - val_loss: 0.1802
